# UAV-SVI student workflow
Run top to bottom from the repository root. Pairs are always built for **one specific OSM building** (UAV chip: that building's footprint; SVI chip: the viewing angle from the camera to that building). Finding good buildings is your job: check every pair visually before labelling.

In [ ]:
import os, sys, subprocess
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, 'scripts')
%pip install -q -r requirements.txt ipywidgets matplotlib

def run(script, *args):
    cmd = [sys.executable, f'scripts/{script}', *map(str, args)]
    print('$', ' '.join(cmd))
    subprocess.run(cmd, check=True)

## 1. Load your package and choose scenes

In [ ]:
import geopandas as gpd
NAME = 'your_name'                                     # letters, digits, _ and - only
PACKAGE = 'data/input/packages/student_01_scenes.gpkg'    # the package you claimed in the Google Sheet

scenes = gpd.read_file(PACKAGE)
scenes['scene_id'] = scenes['scene_id'].astype(str)
display(scenes.drop(columns='geometry')[['scene_id', 'country', 'title', 'gsd_cm', 'area_km2', 'valid_los_matches']])
scenes.plot(column='valid_los_matches', legend=True, figsize=(8, 5));

In [ ]:
# OAM scene ids to work on. Overlapping scenes can be merged in step 2b.
SCENE_IDS = []   # e.g. ['611cce7438f25a0006989180']

## 2. Select scenes and download OSM buildings (ways and relations)

In [ ]:
run('00e_prepare_selection.py', '--package', PACKAGE, '--scene-ids', *SCENE_IDS)

### 2b. Optional: merge overlapping scenes
Only if a building you want is cut by a scene border. Needs at least two scene ids from `SCENE_IDS`.

In [ ]:
MERGE = False
if MERGE:
    run('00f_merge_overlaps.py', '--scene-ids', *SCENE_IDS)

## 3. Parameters you can tweak

In [ ]:
LIMIT = 40                # random sample of buildings for a first run (None = all; each costs a panorama download)
SEARCH_RADIUS_M = 30      # max camera-to-building distance for automatic SVI matching
PADDING_FACTOR = 1.25     # UAV chip size relative to the building
SVI_WINDOW_PX = 1536      # width of the panorama window around the building direction
FOCUS_RATIO = 0.6         # share of that window left unmasked
MANUAL_PAIRS = None       # e.g. 'submission/selected_pairs.csv' from the QGIS plugin (see README)
ONLY_MANUAL = False       # True = only build pairs for buildings listed in MANUAL_PAIRS

## 4. Create the pairs (needs `MAPILLARY_ACCESS_TOKEN` in `.env`)

In [ ]:
run('01_download_uav.py', *(['--limit', LIMIT] if LIMIT else []))
svi_args = ['--search-radius', SEARCH_RADIUS_M]
if MANUAL_PAIRS:
    svi_args += ['--manual-pairs', MANUAL_PAIRS] + (['--only-manual'] if ONLY_MANUAL else [])
run('02_download_svi.py', *svi_args)
run('03_mask_uav.py', '--padding-factor', PADDING_FACTOR)
run('04_mask_svi.py', '--window-width', SVI_WINDOW_PX, '--focus-ratio', FOCUS_RATIO)
run('05_package_pairs.py', '--name', NAME)

### 4b. Look at the pairs first
Drop pairs where the building is not clearly the one in both views (delete `submission/uav/<id>.png`, `submission/svi/<id>.png` and the CSV row, or re-run step 4 with different parameters).

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image
idx = pd.read_csv(f'submission/pairs_index_{NAME}.csv', dtype=str)
for r in idx.head(12).itertuples():
    fig, ax = plt.subplots(1, 2, figsize=(6, 3))
    for a, v in zip(ax, ('uav', 'svi')):
        a.imshow(Image.open(f'submission/{v}/{r.osm_id}.png')); a.axis('off')
    fig.suptitle(f'{r.osm_id}  mapillary {r.mapillary_id}  {r.distance_m} m', fontsize=8)
plt.show()

## 5. Label (option A: here in the notebook)
Option B: the **Label** tab in the QGIS plugin. Option C: Label Studio (see README). All write the same `submission/labels_<NAME>.csv`.

In [ ]:
import ipywidgets as w
from IPython.display import display, clear_output
from utils.labels import VOCAB, LABEL_COLUMNS
csv = f'submission/labels_{NAME}.csv'
df = pd.read_csv(csv, dtype=str, keep_default_na=False)
todo = list(df.index[(df[LABEL_COLUMNS] == '').any(axis=1)])
dropdowns = {c: w.Dropdown(options=VOCAB[c], description=c, style={'description_width': '150px'}) for c in LABEL_COLUMNS}
out = w.Output()
def show():
    with out:
        clear_output(wait=True)
        if not todo:
            print('All pairs labelled.'); return
        oid = df.at[todo[0], 'osm_id']
        print(f'{oid}  ({len(todo)} left)')
        display(w.HBox([w.Image(value=open(f'submission/{v}/{oid}.png', 'rb').read(), width=384) for v in ('uav', 'svi')]))
def save(_):
    i = todo.pop(0)
    for c, d in dropdowns.items():
        df.at[i, c] = d.value
    df.to_csv(csv, index=False)
    show()
btn = w.Button(description='Save and next', button_style='success'); btn.on_click(save)
display(w.VBox([out, *dropdowns.values(), btn])); show()

## 6. Validate before uploading

In [ ]:
run('06_validate_submission.py', '--name', NAME)

## Track B (optional): single-view and cross-view baseline

In [ ]:
%pip install -q -r requirements-model.txt
for view in ['uav', 'svi', 'both']:
    print('====', view)
    subprocess.run([sys.executable, 'model/train.py', '--name', NAME, '--target', 'material_rooftop', '--view', view])